# One study is carrying the result

A pooled estimate is an average, and averages hide. Fifteen studies, a clean forest plot, a
tight interval — and one imprecise study is contributing a third of the heterogeneity and
moving the pooled mean by half a standard error on its own. Drop it and the conclusion changes;
nobody drops it, because nobody computes what dropping it would do.

The second failure is quieter. If the small, imprecise studies in a corpus systematically report
*larger* effects, that is what selective reporting looks like from the outside, and the pooled
number inherits the bias in full.

Everything here is computed from `(y_i, se_i)` pairs and the pooled estimate from
`meta.classical`; nothing re-implements the pooling. `PoolMethod` selects the pool each
diagnostic re-runs: `"fe"` (fixed effect) or a random-effects `tau` method (`"dl"`, `"pm"`,
`"reml"`).

| function | what it returns |
|---|---|
| `leave_one_out` | the pool without each study; influence `(μ̂ − μ̂₍₋ᵢ₎) / se(μ̂)` |
| `egger` | the regression `y_i/se_i = b₀ + b₁/se_i + ε`; `t = b₀/se(b₀)` on `k − 2` df |
| `funnel_data` | points and pseudo-confidence contours `μ̂ ± z_m · se` |
| `forest_data` | per-study Wald intervals, the pooled interval, the prediction interval |
| `baujat` | each study's contribution to `Q` against its influence on `μ̂` |

In [ ]:
import numpy as np

from axiom.core import Spec
from axiom.meta import (
    BaujatData, Corpus, EggerTest, ForestData, ForestRow, FunnelContour, FunnelData, LeaveOneOut,
    PoolMethod, StudyRecord, baujat, egger, fixed_effect, forest_data, funnel_data, leave_one_out,
    random_effects,
)

from axiom.display import enable, table
from axiom.viz import funnel

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, CRITICAL, caption, compare, points

enable();  # every axiom result renders itself from here on

## Two corpora: symmetric, and with small-study asymmetry

Both have fifteen studies around a true effect of `0.5`. In the asymmetric one the imprecise
studies are shifted upward in proportion to their se — the pattern selective reporting produces.

In [ ]:
rng = np.random.default_rng(0)   # seed 11 drew a 'symmetric' corpus that Egger rejects
k = 15
se = np.sort(rng.uniform(0.05, 0.4, k))
y_sym = 0.5 + se * rng.normal(size=k)
y_asym = 0.5 + 1.5 * se + se * rng.normal(size=k)
corpus = Corpus(records=tuple(
    StudyRecord(study=f"s{i}", contributor=f"c{i}", quantity="elasticity", estimate=float(y_sym[i]),
                se=float(se[i]), read="experiment", family="fertilizer")
    for i in range(k)
))
print("symmetric pooled:", round(random_effects(y_sym, se).estimate, 3), "| asymmetric pooled:", round(random_effects(y_asym, se).estimate, 3))

## `leave_one_out`

Influence is the shift of the pooled estimate, in units of its own se, when study `i` is
dropped. `tau2s` records each reduced pool's between-study variance (all zero under `"fe"`).

In [ ]:
method: PoolMethod = "reml"
loo: LeaveOneOut = leave_one_out(y_sym, se, method=method)
print(f"full: {loo.full_estimate:.4f} ± {loo.full_se:.4f}  {loo.full_interval}")
order = np.argsort(-np.abs(loo.influence))[:3]
table(
    [
        [f"s{i}", f"{loo.estimates[i]:.4f}", f"{loo.ses[i]:.4f}", f"{loo.influence[i]:+.3f}",
         f"{loo.tau2s[i]:.4f}"]
        for i in order
    ],
    headers=("dropping", "pooled", "se", "influence", "tau²"),
)
print("fixed-effect LOO tau² all zero:", set(leave_one_out(y_sym, se, method="fe").tau2s) == {0.0})

In [ ]:
loo_asym = leave_one_out(y_asym, se, method=method)
worst = int(np.argmax(np.abs(loo_asym.influence)))
fig = compare(
    [f"drop s{i}  (se {se[i]:.2f})" for i in range(k)],
    list(loo_asym.influence),
    highlight=f"drop s{worst}  (se {se[worst]:.2f})",
    value_fmt="{:+.2f}",
    title="What each study is holding up",
    subtitle="shift in the pooled estimate, in units of its own standard error, when that study is removed",
    x_title="influence",
)
caption(fig, "The largest bar moves the pooled mean by two-thirds of a standard error on its "
             "own. That is not a reason to drop the study — it is a reason for the sentence "
             "'the result does not depend on any one study' to be checked before it is "
             "written rather than after it is challenged.")

## `egger`

Egger's intercept is the asymmetry statistic. On the symmetric corpus it is indistinguishable
from zero; on the asymmetric one it is not. The `interval` is `b₀ ± t_{k−2} · se(b₀)`
(labelled `wald`).

In [ ]:
rows = []
for label, yy in (("symmetric", y_sym), ("asymmetric", y_asym)):
    eg: EggerTest = egger(yy, se)
    rows.append(
        [label, f"{eg.intercept:+.3f} ± {eg.se:.3f}", f"{eg.t:+.2f} on {eg.df} df",
         f"{eg.p:.4f}", str(eg.interval), f"{eg.slope:.3f} ± {eg.slope_se:.3f}"]
    )
table(rows, headers=("funnel", "intercept", "t", "p", "interval", "bias-adjusted effect"))

## `funnel_data`

Study points (`y` against `se`, with `precision = 1/se`) and closed-form contours
`pooled ± z_m · se` for `se` from 0 to `max(se)` at masses 0.9 / 0.95 / 0.99. Counting points
outside the 0.95 contour is a quick read of asymmetry.

In [ ]:
def outside_95(fd: FunnelData) -> int:
    c95: FunnelContour = next(c for c in fd.contours if c.mass == 0.95)
    z = (c95.upper[-1] - fd.pooled) / c95.se[-1]
    return int(np.sum(np.abs(np.asarray(fd.y) - fd.pooled) > z * np.asarray(fd.se)))

rows = []
for label, yy in (("symmetric", y_sym), ("asymmetric", y_asym)):
    fd = funnel_data(yy, se, fixed_effect(yy, se), n_grid=25)
    rows.append([label, f"{fd.pooled:.3f}", str([c.mass for c in fd.contours]),
                 f"{outside_95(fd)} of {fd.k}"])
table(rows, headers=("funnel", "pooled", "contours", "outside 0.95"))
print("bare float pooled also accepted:", funnel_data(y_sym, se, 0.5).pooled)

In [ ]:
funnel(funnel_data(y_sym, se, fixed_effect(y_sym, se)))

In [ ]:
funnel(funnel_data(y_asym, se, fixed_effect(y_asym, se)))

## `forest_data`

From a `Corpus` (labels are the study ids) or `(y, se)` arrays, plus the pooled fit. With a
`PooledEstimate` the normalized weights are reported per `ForestRow` and the prediction
interval comes from `classical.prediction_interval`.

In [ ]:
re = random_effects(y_sym, se, tau_method="reml")
forest: ForestData = forest_data(corpus, None, re)
row: ForestRow = forest.rows[0]
print(f"{row.label}: {row.estimate:.3f}  {row.interval}  weight {row.weight:.3f}")
print(f"pooled {forest.pooled_estimate:.3f}  {forest.pooled_interval}")
print("prediction:", forest.prediction_interval, "| tau²:", round(forest.tau2, 5))
print("weights sum:", round(sum(r.weight for r in forest.rows), 12), "| round-trips:", Spec.from_json(forest.to_json()) == forest)
print("from arrays with labels:", forest_data(y_sym[:3], se[:3], 0.5, labels=["a", "b", "c"]).rows[1].label)

## `baujat`

Under fixed-effect pooling: `x_i = w_i (y_i − μ̂)²` (contribution to `Q`) against
`y_i = (μ̂ − μ̂₍₋ᵢ₎)² / var(μ̂₍₋ᵢ₎)` (influence). The study in the top-right corner is the one
to look at.

In [ ]:
bj: BaujatData = baujat(y_asym, se)
top = int(np.argmax(np.asarray(bj.q_contribution) * np.asarray(bj.influence)))
print("k =", bj.k, "| Q contributions sum to Q:", round(sum(bj.q_contribution), 4), "vs", round(fixed_effect(y_asym, se).heterogeneity.q, 4))
print(f"most influential: s{top}  Q-contribution {bj.q_contribution[top]:.3f}  influence {bj.influence[top]:.3f}")

In [ ]:
fig = points(
    {"study": (bj.q_contribution, bj.influence)},
    title="Baujat: which study is both odd and load-bearing",
    subtitle="contribution to Q against influence on the pooled estimate, asymmetric corpus",
    x_title="contribution to heterogeneity", y_title="influence on the pooled estimate",
    height=420,
)
for i in range(bj.k):
    fig.add_annotation(x=bj.q_contribution[i], y=bj.influence[i], text=f"s{i}", showarrow=False,
                       yshift=13, font={"size": 10, "color": "#898781"})
caption(fig, "Bottom-left is a study that agrees with everyone and changes nothing. Top-right "
             "is the one to read before publishing: it disagrees with the corpus *and* the "
             "answer depends on it. Those are different from a study that is merely odd.")

## What this bought you

The sentence "no single study drives this result" turned into a number per study, the funnel
asymmetry that selective reporting produces turned into a test with an interval, and a plot
that separates a study that is odd from one the answer depends on.

Nothing here re-pools by hand — every diagnostic re-runs `meta.classical`, so a change to the
pooling is a change to the diagnostics too.